### Initialize libraries, functions and MiP-CRIP solver

In [1]:
import networkx as nx
import pandas as pd
import numpy as np
import time
from mip_crip import MiP_CRIP, eng2cut, data2graph


### Sherrington-Kirkpatrick (SK) Model

In [2]:
# Generate SK model
n = 1000  # number of spins
J_mat = np.random.randn(n, n).astype(np.float32)
J_mat = (J_mat + J_mat.T) / 2  # Make it symmetric
np.fill_diagonal(J_mat, 0)  # Zero diagonal

print(f"SK model generated with {n} spins.")

SK model generated with 1000 spins.


### Complete Graph ($K_n$)

In [3]:
# Generate Ising model from K_n graph
n = 2000  # number of spins
J_mat = np.random.randn(n, n).astype(np.float32)
J_mat = J_mat + J_mat.T  # Make it symmetric
J_mat = np.sign(J_mat)  # Ensure entries are -1 or +1
np.fill_diagonal(J_mat, 0)  # Zero diagonal

print(f"Complete graph Ising model generated with {n} spins.")

Complete graph Ising model generated with 2000 spins.


### G-set graph

In [4]:
# Generate Ising model from G10 graph for MAX-CUT problem
G, Adj_mat = data2graph('G10_graph.txt')  # Load the complex G-set graph G10
J_mat = -Adj_mat
n = J_mat.shape[0]

print(f"G10 graph loaded with {n} spins.")

G10 graph loaded with 800 spins.


### Number Partitioning Problem (NPP)

In [5]:
# Generate Ising model for uniform NPP
n = 100
N_vec = np.random.rand(n).flatten()
J_mat = - np.outer(N_vec, N_vec)
np.fill_diagonal(J_mat, 0)

print(f"Uniform NPP Ising model generated with {n} spins.")

Uniform NPP Ising model generated with 100 spins.


### Run MiP-CRIP

In [7]:
# initialize hyperparameters for tuning

rounds = 10      # tuning rounds
tune_res = 3     # tuning grid-resolution

K_outer = 200   # epochs
T_inner = 10    # inner iterations for ADAM optimizer

gamma0 = 0.1
beta0 = 10
step0 = 1

sigma_noise = 1e-3

best_energy = np.inf
best_cut = 0
best_sync = 0
best_params = None

n = J_mat.shape[0]
J_sum = np.sum(J_mat)
x0 = np.random.randn(n)

Steps = np.linspace(0, step0, tune_res)[1:]
Gamma = np.linspace(0, gamma0, tune_res)[1:]
Beta = np.linspace(0, beta0, tune_res)[1:]

for round_num in range(rounds):
    for k, step in enumerate(Steps):
        for i, gamma in enumerate(Gamma):
            for j, beta in enumerate(Beta):

                lambda0 = np.sqrt(gamma / (2 * beta))

                for lambda_ in np.linspace(0, lambda0, tune_res)[1:]:
                    for alpha in np.linspace(3 * beta * lambda_**2, beta * lambda_**2 + gamma, tune_res):

                        start_time = time.time()

                        spin_vec = MiP_CRIP(J_mat, x0, T=T_inner, K=K_outer,
                                                alpha=alpha, beta=beta, lambda_=lambda_,
                                                step=step,          # Adam learning rate
                                                beta1=0.09, beta2=0.999, eps=1e-8,
                                                sigma_noise=sigma_noise,
                                                rng=None, return_all=False)
                        
                        t_elapsed = time.time() - start_time
                        
                        energy = -0.5 * spin_vec.T @ J_mat @ spin_vec
                        sync = np.mean(spin_vec == np.sign(J_mat @ spin_vec))
                        cut_value = eng2cut(energy, J_sum)
                        print(f"Round: {round_num+1}/{rounds} | alpha: {alpha:.4f} beta: {beta:.4f} gamma: {gamma:.4f} lambda: {lambda_:.4f} | learning rate: {step:.4f} | "
                              f"energy: {energy:.2f} (best:{best_energy:.2f}) | cut: {cut_value:.2f} (best:{best_cut:.2f}) | sync: {sync:.3f} (best:{best_sync:.3f}) | runtime: {t_elapsed*1000:.1f}ms", end='\r')

                        if energy < best_energy:
                            best_spin = spin_vec.copy()
                            best_energy = energy
                            best_cut = cut_value
                            best_sync = sync
                            best_params = {
                                "alpha": alpha,
                                "gamma": gamma,
                                "beta": beta,
                                "lambda_": lambda_,
                                "step": step,
                                'i': i,
                                'j': j,
                                'k': k
                            }

    # grid refinement
    if best_params is None:
        continue

    i, j, k = best_params["i"], best_params["j"], best_params["k"]

    # refine gamma
    if i == 0:
        Gamma = np.linspace(max(1e-6, Gamma[i]/2), Gamma[i], tune_res)
    elif i == len(Gamma) - 1:
        Gamma = np.linspace(Gamma[i], 2 * Gamma[i], tune_res)
    else:
        Gamma = np.linspace(Gamma[i - 1], Gamma[i + 1], tune_res)
    
    # refine beta
    if j == 0:
        Beta = np.linspace(max(1e-6, Beta[j]/2), Beta[j], tune_res)
    elif j == len(Beta) - 1:
        Beta = np.linspace(Beta[j], 2 * Beta[j], tune_res)
    else:
        Beta = np.linspace(Beta[j - 1], Beta[j + 1], tune_res)

    # refine step (learning rate)
    if k == 0:
        Steps = np.linspace(max(1e-6, Steps[k]/2), Steps[k], tune_res)
    elif k == len(Steps) - 1:
        Steps = np.linspace(Steps[k], 2 * Steps[k], tune_res)
    else:
        Steps = np.linspace(Steps[k - 1], Steps[k + 1], tune_res)

    print("\n")

print(f"Tuned parameters: alpha: {best_params['alpha']:.4f}, beta: {best_params['beta']:.4f}, gamma: {best_params['gamma']:.4f}, lambda={best_params['lambda_']:.4f}, learning rate: {best_params['step']:.4f}")


Round: 1/10 | alpha: 0.1500 beta: 10.0000 gamma: 0.1000 lambda: 0.0707 | learning rate: 1.0000 | energy: 1082.17 (best:981.48) | cut: 74.96 (best:125.30) | sync: 0.070 (best:0.100) | runtime: 23.0ms

Round: 2/10 | alpha: 0.0750 beta: 20.0000 gamma: 0.0500 lambda: 0.0354 | learning rate: 0.5000 | energy: 706.33 (best:609.40) | cut: 262.88 (best:311.34) | sync: 0.160 (best:0.200) | runtime: 22.1mss

Round: 3/10 | alpha: 0.0375 beta: 20.0000 gamma: 0.0250 lambda: 0.0250 | learning rate: 0.2500 | energy: 609.40 (best:306.34) | cut: 311.34 (best:462.88) | sync: 0.200 (best:0.280) | runtime: 22.4mss

Round: 4/10 | alpha: 0.0188 beta: 40.0000 gamma: 0.0125 lambda: 0.0125 | learning rate: 0.1250 | energy: 195.39 (best:-17.40) | cut: 518.35 (best:624.74) | sync: 0.310 (best:0.670) | runtime: 21.8mss

Round: 5/10 | alpha: 0.0094 beta: 20.0000 gamma: 0.0063 lambda: 0.0125 | learning rate: 0.0625 | energy: -16.59 (best:-17.58) | cut: 624.34 (best:624.84) | sync: 0.460 (best:1.000) | runtime: 23.2m

### Show results

In [8]:
print(f"Best energy: {best_energy:.3f}, Best cut: {best_cut:.3f}, Best sync: {best_sync:.3f}")

# uncomment for NPP discrepancy results
print(f"Best Discrepancy: {np.abs(best_spin @ N_vec):.5f}, Best sync: {best_sync:.3f}")


Best energy: -17.584, Best cut: 624.836, Best sync: 1.000
Best Discrepancy: 0.00001, Best sync: 1.000
